<a href="https://colab.research.google.com/github/yianchen903/my-project/blob/main/LDG_Mimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def vecnorm(u):
  absu = pow(inner(u,u),0.5)
  return absu

def scalarnorm(u):
  absu = pow(pow(u,2),0.5)
  return absu


# DG discrete weak gradient operators
def Discrete_weak_gradient(v,g):
  (q,sigma,uu) = TrialFunctions(W)
  (zeta,tau,vv) = TestFunctions(W)
  # v = project(v,V3) # v function
  # g = project(g,V3) # g function

  # # choose C12
  C12 = Constant((1.0,0))
  dot_product = inner(C12,n('+'))
  sign = conditional(dot_product > 0, 1.0, -1.0)

  bc = DirichletBC(W.sub(2), g, "on_boundary")

  L = inner(q,zeta)*dx
  # R = -v*div(zeta)*dx + avg(v)*inner(jump(zeta),n('+'))*dS\
  #   + inner(C12,jump(v,n))*inner(jump(zeta),n('+'))*dS + g*inner(zeta,n)*ds
  R = -v*div(zeta)*dx + avg(v)*inner(jump(zeta),n('+'))*dS\
    + inner(0.5*sign,jump(v))*inner(jump(zeta),n('+'))*dS + g*inner(zeta,n)*ds

  F = L - R
  L = lhs(F)
  R = rhs(F)

  w = Function(W)
  solve(L==R,w,bc)
  (q,sigma,u) = w.split()

  return q

# energy functional
def J(u,g,eta):
  eta = Constant(eta)
  Dwg_u_g = Discrete_weak_gradient(u,g)
  Ju =  1/p*pow(vecnorm(Dwg_u_g),p)*dx\
     + 1/p*eta*pow(h_avg,1-p)*pow(vecnorm(jump(u,n)),p)*dS\
     + 1/p*eta*pow(h,1-p)*pow(scalarnorm(u-g),p)*ds - f*u*dx

  return Ju

# nonlinear coefficient
def nonlinear(vec):
  nonlinear = pow((vecnorm(vec)),p-2)*vec
  return nonlinear


# calculate derivative of energy functional
def dJudv_value(u,g,v,eta,Dwg1,Dwg2): # v function
  eta = Constant(eta)
  nonlinear1 = nonlinear(Dwg1)
  nonlinear2 = nonlinear((1/h_avg)*jump(u,n))
  nonlinear3 = nonlinear((1/h)*(u-g)*n)

  dJudv_value = inner(nonlinear1,Dwg2)*dx\
            + eta*inner(nonlinear2,jump(v,n))*dS\
            + eta*inner(nonlinear3,v*n)*ds - f*v*dx

  return assemble(dJudv_value)


# derivative of energy functional
def dJudv_fn(u,g,v,eta): # v testfunction
  eta = Constant(eta)
  nonlinear2 = nonlinear((1/h_avg)*jump(u,n))
  nonlinear3 = nonlinear((1/h)*(u-g)*n)

  dJudv_fn =  eta*inner(nonlinear2,jump(v,n))*dS\
            + eta*inner(nonlinear3,v*n)*ds - f*v*dx

  return dJudv_fn


# ---- initial guess----
eta = 10.0
uD = Constant(0.0) # Dirichlet boundary
uex = project(uex,V3)
uD = uex
Juex_value = assemble(J(uex,uD,eta))

# initial guess

u0 = uex
# u0 = project(Expression("1-r ", degree=5,r=r), V3)
u0 = project(Expression("(x[0]-1)*x[0]*(x[1]-1)*x[1]", degree=5), V3)

# set Amat, bvec, Dwg_v_0, degree of freedoms

Amat = np.zeros((V3.dim(), V3.dim()))
bvec = np.zeros(V3.dim())
dofmap = V3.dofmap()
global_dofs = dofmap.dofs()

Dwg_phi_j_0_list = []
for j in range(V3.dim()):
    phi_j = Function(V3)
    phi_j.vector()[j] = 1
    # print("phi_j=",phi_j.vector().get_local())
    Dwg_phi_j_0_list.append(Discrete_weak_gradient(phi_j, Constant(0.0)))

print("Dimension V3: ",V3.dim())

# set boundary condition and collect boundary dofs

uD = project(uD,V3)
bc = DirichletBC(V3, uD, "on_boundary")
boundary_dofs = list(bc.get_boundary_values().keys())

eps = 1.0
tol = 1.0e-6
iter = 0
maxiter = 10
epsi = 0.0001 # avoid degeneracy

while eps > tol and iter < maxiter:

  w = TrialFunction(V3)
  v = TestFunction(V3)
  alpha = 1.0

# Find the steepest direction
# The first term of LHS and RHS for p=2

  Dwg_u0_g = Discrete_weak_gradient(u0,uD)
  for i in range(V3.dim()):
    Dwg_phi_i_0 = Dwg_phi_j_0_list[i]
    phivalue = assemble(inner(Dwg_u0_g, Dwg_phi_i_0) * dx)
    bvec[global_dofs[i]] = phivalue


    for j in range(V3.dim()):
      Dwg_phi_j_0 = Dwg_phi_j_0_list[j]
      phi_ij_value = assemble(inner(Dwg_phi_i_0, Dwg_phi_j_0) * dx)
      Amat[global_dofs[i], global_dofs[j]] = phi_ij_value

  # print("Amat: ",Amat)
  # print("bvec: ",bvec)

# method 1. convert the other terms of LHS and RHS to numpy and add them, respectively.
# method 2. apply boundary to the other terms

  weighted = assemble(Constant(eta)*(1/h_avg)*inner(jump(w,n),jump(v,n))*dS\
            + Constant(eta)*(1/h)*w*v*ds)
  # bc.apply(weighted) # method 2
  weighted = as_backend_type(weighted).mat() # method 1
  weighted = weighted.getValues(range(0,weighted.getSize()[0]), range(0,weighted.getSize()[1])) # method 1
  Amat = Amat + weighted # method 1
  # print("Amat =",Amat)


  dJudv = assemble(dJudv_fn(u0,uD,v,eta))
  # bc.apply(dJudv) # method 2
  dJudv_vec = dJudv.get_local() # method 1
  bvec = bvec + dJudv_vec # method 1
  # print("bvec =", bvec)


  # for dof in boundary_dofs:
  #   Amat[dof] = 0
  #   # Amat[dof, dof] = 1 # method 1
  #   bvec[dof] = 0


# convert numpy matrix to dolfin.cpp.la.PETScMatrix
  AMat = PETSc.Mat().createAIJ(Amat.shape)
  AMat.setUp()
  AMat.setValues(range(0, Amat.shape[0]), range(0, Amat.shape[1]), Amat)
  AMat.assemble()
  AMat = PETScMatrix(AMat)

  # AMat = AMat + weighted # method 2 # add the first and the other terms of LHS


# convert numpy vector to dolfin.cpp.la.PETScVector
  bVec = PETSc.Vec().createSeq(V3.dim())
  bVec.setValues(range(0,V3.dim()), bvec)
  bVec = PETScVector(bVec)

  # bVec = bVec + dJudv # method 2 # add the first and the other terms of RHS

  w = Function(V3)
  solve(AMat, w.vector(), -bVec)

  # print("w=",w.vector().get_local()[:])
# doing Backtrack line search

  tol_l = 1e-4
  c2 = 1e-4
  c1 = 0.6
  stepsize = alpha
  starting = u0
  direction = w

  Jw_value = assemble(J(w,uD,eta))
  Jz0_value = assemble(J(u0,uD,eta))
  Dwg_u0_g = Discrete_weak_gradient(direction,uD)
  Dwg_dire_0 = Discrete_weak_gradient(direction,Constant(0.0))
  dJudz0 = dJudv_value(starting,uD,direction,eta,Dwg_u0_g,Dwg_dire_0)

  zn = Function(V3)
  iter_l = 0

  while stepsize>tol_l:
    print("****line search iter=%d: stepsize=%.16f ****"%(iter_l,stepsize))
    iter_l  = iter_l + 1
    zn = starting+stepsize*direction
    Jzn_value = assemble(J(zn,uD,eta))
    if Jzn_value < Jz0_value+c2*stepsize*dJudz0:
      break
    else:
      stepsize = c1*stepsize

  print("After doing line search: ")
  print("iter_stepsize=%d:" %iter_l)
  print ("stepsize=%.16f Jz0=%.16f Jw=%.16f Jzn=%.16f Juex=%.16f" %(stepsize, Jz0_value, Jw_value, Jzn_value, Juex_value))

  alpha = stepsize
  u = u0 + alpha * w

  u_er = project(w, V3)
  u_exer = project(u-uex, V3)
  u_L2iter = sqrt(assemble(inner(u_er,u_er)*dx))
  uex_L2iter = sqrt(assemble(inner(u_exer,u_exer)*dx))
  eps = u_L2iter
  eps_ex =  uex_L2iter
  print ("iter=%d: error_iter=%.16f error_ex=%.16f" %(iter, eps, eps_ex))
  print (" ")

  u0 = u
  iter = iter + 1
